In [1]:
import pandas as pd
import math
from pathlib import Path

# 1. Paths and Data Loading
BASE_DIR = Path.cwd().parent
if not (BASE_DIR / "data" / "processed").exists():
    BASE_DIR = Path.cwd()

ELO_PATH = BASE_DIR / "data" / "processed" / "current_elo_ratings.csv"
OUTPUT_PATH = BASE_DIR / "data" / "processed" / "most_likely_path.txt"

df_elo = pd.read_csv(ELO_PATH)
elo_dict = dict(zip(df_elo['team'], df_elo['current_elo']))

# 2. Math Engine with Reality Check (Wstrzykiwanie wyników z boiska)
# Stan na dzień dzisiejszy: cała runda 1/16 finału (Round of 32) ORAZ 6 z 8
# meczów 1/8 finału (Round of 16) zostały już rozegrane w rzeczywistości.
#
# WAŻNE - twardy "override": predict_match() ZAWSZE zwraca zwycięzcę
# zapisanego w known_winners, całkowicie ignorując to, co mówi model ELO,
# nawet jeśli ta drużyna miała niższe procentowe szanse (upset). ELO decyduje
# wyłącznie o meczach, które faktycznie jeszcze się nie odbyły.
# Klucze sa frozenset({team_a, team_b}) - dopasowanie meczu jest z definicji
# NIEZALEZNE od kolejnosci drużyn (frozenset({"A","B"}) == frozenset({"B","A"})).
known_winners = {
    frozenset({'Colombia', 'Ghana'}): 'Colombia',
    frozenset({'Argentina', 'Cape Verde'}): 'Argentina',
    frozenset({'Egypt', 'Australia'}): 'Egypt',
    frozenset({'Switzerland', 'Algeria'}): 'Switzerland',
    frozenset({'Portugal', 'Croatia'}): 'Portugal',
    frozenset({'Spain', 'Austria'}): 'Spain',
    frozenset({'United States', 'Bosnia and Herzegovina'}): 'United States',
    frozenset({'Belgium', 'Senegal'}): 'Belgium',
    frozenset({'England', 'DR Congo'}): 'England',
    frozenset({'Mexico', 'Ecuador'}): 'Mexico',
    frozenset({'France', 'Sweden'}): 'France',
    frozenset({'Norway', 'Ivory Coast'}): 'Norway',
    frozenset({'Morocco', 'Netherlands'}): 'Morocco',
    frozenset({'Paraguay', 'Germany'}): 'Paraguay',
    frozenset({'Brazil', 'Japan'}): 'Brazil',
    frozenset({'Canada', 'South Africa'}): 'Canada',
    frozenset({'Belgium', 'United States'}): 'Belgium',
    frozenset({'Spain', 'Portugal'}): 'Spain',
    frozenset({'England', 'Mexico'}): 'England',
    frozenset({'Norway', 'Brazil'}): 'Norway',
    frozenset({'France', 'Paraguay'}): 'France',
    frozenset({'Morocco', 'Canada'}): 'Morocco',
    frozenset({'Argentina', 'Egypt'}): 'Argentina',
    frozenset({'Switzerland', 'Colombia'}): 'Switzerland',
    frozenset({'France', 'Morocco'}): 'France'
}

def predict_match(team_a, team_b):
    # ELO liczone jest zawsze, nawet dla meczów już rozegranych - żeby było
    # widać, jakie szanse dawał model PRZED meczem.
    elo_a = elo_dict.get(team_a, 1500)
    elo_b = elo_dict.get(team_b, 1500)
    prob_a = 1.0 / (1.0 + math.pow(10, (elo_b - elo_a) / 400.0))
    prob_b = 1.0 - prob_a

    # 1. Sprawdzenie, czy mecz już się odbył w rzeczywistości -> TWARDY
    #    OVERRIDE: rzeczywisty wynik wygrywa bezwzględnie z modelem ELO.
    #    Dopasowanie jest niezależne od kolejności drużyn (frozenset klucz).
    winner = known_winners.get(frozenset({team_a, team_b}))

    # Zabezpieczenie (nie powinno się zdarzyć przy poprawnych danych): jeśli
    # wpis w known_winners wskazuje zwycięzcę, który nie jest żadną z dwóch
    # drużyn tego meczu, NIE przerywamy działania - po prostu ignorujemy ten
    # (błędny) wpis i pozwalamy zdecydować modelowi ELO.
    if winner is not None and winner not in (team_a, team_b):
        winner = None

    if winner is not None:
        return winner, prob_a * 100, prob_b * 100, True # True = to fakt historyczny

    # 2. Jeśli mecz przed nami -> decyduje matematyka ELO
    winner = team_a if prob_a > 0.5 else team_b
    return winner, prob_a * 100, prob_b * 100, False

# 3. FULL Round of 32 Topology
left_bracket_r32 = [
    ("Germany", "Paraguay"),
    ("France", "Sweden"),
    ("South Africa", "Canada"),
    ("Netherlands", "Morocco"),
    ("Portugal", "Croatia"),
    ("Spain", "Austria"),
    ("United States", "Bosnia and Herzegovina"),
    ("Belgium", "Senegal")
]

right_bracket_r32 = [
    ("Brazil", "Japan"),
    ("Ivory Coast", "Norway"),
    ("Mexico", "Ecuador"),
    ("England", "DR Congo"),
    ("Argentina", "Cape Verde"),
    ("Australia", "Egypt"),
    ("Switzerland", "Algeria"),
    ("Colombia", "Ghana")
]

report = []
report.append("🏆 WORLD CUP 2026: FULL 32-TEAM DETERMINISTIC PATH 🏆\n")
report.append("="*65)

def format_match_line(label, team_a, p_a, team_b, p_b, winner, is_real):
    # Nawet dla meczów już rozegranych (is_real) procenty ELO są pokazywane -
    # to szanse, jakie model dawał PRZED rozegraniem meczu; sam wynik i tak
    # bierze się bezwzględnie z known_winners (twardy override).
    tag = " (ACTUAL RESULT)" if is_real else ""
    return f"{label} {team_a} ({p_a:.1f}%) vs {team_b} ({p_b:.1f}%) ---> {winner}{tag}"

def simulate_side(bracket_pairs, side_name):
    report.append(f"\n--- {side_name} BRACKET ---")
    
    # Round of 32
    r16_teams = []
    for team_a, team_b in bracket_pairs:
        winner, p_a, p_b, is_real = predict_match(team_a, team_b)
        report.append(format_match_line("[R32]", team_a, p_a, team_b, p_b, winner, is_real))
        r16_teams.append(winner)
        
    # Round of 16
    qf_teams = []
    for i in range(0, len(r16_teams), 2):
        team_a, team_b = r16_teams[i], r16_teams[i+1]
        winner, p_a, p_b, is_real = predict_match(team_a, team_b)
        report.append(format_match_line("[R16]", team_a, p_a, team_b, p_b, winner, is_real))
        qf_teams.append(winner)

    # Quarter-Finals
    sf_teams = []
    for i in range(0, len(qf_teams), 2):
        team_a, team_b = qf_teams[i], qf_teams[i+1]
        winner, p_a, p_b, is_real = predict_match(team_a, team_b)
        report.append(format_match_line("[QF] ", team_a, p_a, team_b, p_b, winner, is_real))
        sf_teams.append(winner)
        
    # Semi-Finals
    team_a, team_b = sf_teams[0], sf_teams[1]
    finalist, p_a, p_b, is_real = predict_match(team_a, team_b)
    tag = " (ACTUAL RESULT)" if is_real else ""
    report.append(f"[SF]  {team_a} ({p_a:.1f}%) vs {team_b} ({p_b:.1f}%) ---> {finalist} GOES TO FINAL{tag}")
    
    return finalist

left_finalist = simulate_side(left_bracket_r32, "LEFT")
right_finalist = simulate_side(right_bracket_r32, "RIGHT")

# --- THE GRAND FINAL ---
report.append("\n" + "="*65)
report.append("👑 THE GRAND FINAL 👑")
report.append("="*65)
champion, p_a, p_b, is_real = predict_match(left_finalist, right_finalist)
final_tag = " (ACTUAL RESULT)" if is_real else ""
report.append(f"{left_finalist} (Left) ({p_a:.1f}%) vs {right_finalist} (Right) ({p_b:.1f}%){final_tag}")
report.append(f"\n🏆 PREDICTED WORLD CUP CHAMPION: {champion.upper()} 🏆")

# NOTE: app.py implements a non-fatal version of this sanity check
# (find_unmatched_known_results()) that walks the whole simulated bracket
# and surfaces (as an info note in the sidebar, never as a crash) any
# KNOWN_RESULTS/known_winners entry that was never matched against a real
# fixture.

# Printowanie linijka po linijce omija problem z ucinaniem tekstu w VS Code
for line in report:
    print(line)

with open(OUTPUT_PATH, "w", encoding="utf-8") as file:
    file.write("\n".join(report))

🏆 WORLD CUP 2026: FULL 32-TEAM DETERMINISTIC PATH 🏆


--- LEFT BRACKET ---
[R32] Germany (64.1%) vs Paraguay (35.9%) ---> Paraguay (ACTUAL RESULT)
[R32] France (84.0%) vs Sweden (16.0%) ---> France (ACTUAL RESULT)
[R32] South Africa (34.5%) vs Canada (65.5%) ---> Canada (ACTUAL RESULT)
[R32] Netherlands (41.8%) vs Morocco (58.2%) ---> Morocco (ACTUAL RESULT)
[R32] Portugal (59.8%) vs Croatia (40.2%) ---> Portugal (ACTUAL RESULT)
[R32] Spain (74.2%) vs Austria (25.8%) ---> Spain (ACTUAL RESULT)
[R32] United States (76.8%) vs Bosnia and Herzegovina (23.2%) ---> United States (ACTUAL RESULT)
[R32] Belgium (52.4%) vs Senegal (47.6%) ---> Belgium (ACTUAL RESULT)
[R16] Paraguay (21.9%) vs France (78.1%) ---> France (ACTUAL RESULT)
[R16] Canada (28.6%) vs Morocco (71.4%) ---> Morocco (ACTUAL RESULT)
[R16] Portugal (37.8%) vs Spain (62.2%) ---> Spain (ACTUAL RESULT)
[R16] United States (47.4%) vs Belgium (52.6%) ---> Belgium (ACTUAL RESULT)
[QF]  France (55.2%) vs Morocco (44.8%) ---> France (